##Notebook 7: Raw Text Contract Processing

This notebook builds MeridianIQ's raw TXT contract processing pipeline.

Earlier notebooks used CUAD's structured master clause file to understand contracts, build fingerprints, train clause detection models, retrieve evidence, and score risk. This notebook moves one step closer to a real user-facing system by processing a raw contract text file directly.

The goal is to transform an unstructured TXT contract into structured MeridianIQ outputs: cleaned text, readable chunks, predicted clauses, detected clause summaries, and retrieved supporting evidence.

##Importing Libraries and Locating Raw Contract TXT Folder

In [ ]:
import pandas as pd
import numpy as np
import joblib,json,os,re
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from fingerprint_builder import *

!pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer


from google.colab import drive
drive.mount('/content/drive')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 300)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Verifying the TXT Directory

In [ ]:
txt_dir = "/content/drive/MyDrive/LexSight/CUAD_v1/full_contract_txt"

In [ ]:
print(os.path.exists(txt_dir))

True


##Loading Saved Models and Risk Configuration

In [ ]:
tfidf=joblib.load("tfidf_vectorizer.pkl")
clause_detector=joblib.load("baseline_clause_detector.pkl")

risk_config=pd.read_csv("clause_risk_config.csv")
mvp_config=risk_config[risk_config["is_mvp_clause"]==True].copy()

##Listing Available TXT Contracts

In [ ]:
txt_files=[f for f in os.listdir(txt_dir) if f.endswith(".txt")]
len(txt_files),txt_files[:5]

(510,
 ['ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt',
  'ADUROBIOTECH,INC_06_02_2020-EX-10.7-CONSULTING AGREEMENT.txt',
  'ADMA BioManufacturing, LLC -  Amendment 3 to Manufacturing Agreement.txt',
  'ALLISONTRANSMISSIONHOLDINGSINC_12_15_2014-EX-99.1-COOPERATION AGREEMENT.txt',
  'ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT.txt'])

##Loading a Sample Raw Contract

A sample TXT contract is selected and loaded into memory.

At this stage, the contract is still raw text.

This represents the kind of input MeridianIQ would receive from a user or application before any cleaning, chunking, detection, retrieval, or risk analysis happens.

In [ ]:
sample_txt_path=os.path.join(txt_dir,txt_files[0])

with open(sample_txt_path,"r",encoding="utf-8",errors="ignore") as f:
  raw_text=f.read()

len(raw_text),raw_text[:1000]

(143307,
 'ALAMOGORDO FINANCIAL CORPORATION                                 1,101,643 Shares\n\n                                  COMMON STOCK                            (Par Value $.0l Per Share)\n\n                       Subscription Price $10.00 Per Share\n\n                                AGENCY AGREEMENT\n\n                              ___________ __, 2000\n\nCharles Webb & Company, a Division of Keefe, Bruyette & Woods, Inc. 211 Bradenton Avenue Dublin, Ohio 43017\n\nLadies and Gentlemen:\n\n         Alamogordo   Financial   Corporation,   a  federal   corporation   (the "Company"), AF Mutual Holding Company (the "MHC") and Alamogordo Federal Savings and Loan Association,  a federally  chartered stock savings and loan association (the  "Bank")  with its  deposit  accounts  insured by the  Savings  Association Insurance  Fund  ("SAIF")   administered  by  the  Federal   Deposit   Insurance Corporation  ("FDIC"),  hereby confirm,  jointly and severally,  their agreement with Charl

##Cleaning Contract Text

Raw contract text often contains irregular whitespace, non-breaking spaces, line breaks, and formatting artifacts.

This step cleans the text so that downstream models receive a more consistent input

In [ ]:
def clean_contract_text(text):
    text=text.replace("\xa0", " ")
    text=re.sub(r"\s+", " ", text)
    text=text.strip()
    return text

##Inspecting Cleaned Text

In [ ]:
clean_text=clean_contract_text(raw_text)

len(clean_text),clean_text[:1000]

(113831,
 'ALAMOGORDO FINANCIAL CORPORATION 1,101,643 Shares COMMON STOCK (Par Value $.0l Per Share) Subscription Price $10.00 Per Share AGENCY AGREEMENT ___________ __, 2000 Charles Webb & Company, a Division of Keefe, Bruyette & Woods, Inc. 211 Bradenton Avenue Dublin, Ohio 43017 Ladies and Gentlemen: Alamogordo Financial Corporation, a federal corporation (the "Company"), AF Mutual Holding Company (the "MHC") and Alamogordo Federal Savings and Loan Association, a federally chartered stock savings and loan association (the "Bank") with its deposit accounts insured by the Savings Association Insurance Fund ("SAIF") administered by the Federal Deposit Insurance Corporation ("FDIC"), hereby confirm, jointly and severally, their agreement with Charles Webb & Company, a Division of Keefe, Bruyette & Woods, Inc. (the "Agent"), as follows: Section 1. The Offering. In accordance with the Stock Issuance Plan adopted by its Board of Directors (the "Plan"), the Company will offer and sell up to

##Preparing for Text Chunking

Contracts are usually too long to process as one continuous block for evidence retrieval.

This section begins the chunking process by preparing helper logic for splitting the contract into meaningful text units.

Chunking allows MeridianIQ to later retrieve specific evidence passages instead of returning the entire contract.

In [ ]:
def split_into_paragraphs(text):
    paragraphs = re.split(r"\n\s*\n", text)

    paragraphs = [
        p.strip()
        for p in paragraphs
        if p.strip()
    ]

    return paragraphs

##Paragraph-Aware Chunking

This step splits the contract into paragraph-aware chunks.

The goal is to avoid cutting text in the middle of words or legal sentences.

In [ ]:
def chunk_text(text, max_chars=1200):

    paragraphs = split_into_paragraphs(text)

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        # Paragraph fits into current chunk
        if len(current_chunk) + len(paragraph) + 2 <= max_chars:

            current_chunk += paragraph + "\n\n"

        else:

            # Save previous chunk
            if current_chunk.strip():

                chunks.append(current_chunk.strip())


            if len(paragraph) > max_chars:

                chunks.extend(
                    split_large_paragraph(
                        paragraph,
                        max_chars
                    )
                )

                current_chunk = ""

            else:

                current_chunk = paragraph + "\n\n"

    if current_chunk.strip():

        chunks.append(current_chunk.strip())

    chunk_df = pd.DataFrame({

        "chunk_id": range(len(chunks)),
        "chunk_text": chunks

    })

    return chunk_df

##Handling Large Paragraphs

Some legal paragraphs are very long and may exceed the desired chunk size.

This step handles those cases by splitting large paragraphs into sentence-level chunks.

This keeps the chunks within a manageable size while preserving sentence boundaries as much as possible.

In [ ]:
def split_large_paragraph(paragraph, max_chars):

    sentences = re.split(
        r'(?<=[.!?])\s+',
        paragraph
    )

    chunks = []
    current = ""

    for sentence in sentences:

        if len(current) + len(sentence) + 1 <= max_chars:

            current += sentence + " "

        else:

            chunks.append(current.strip())
            current = sentence + " "

    if current.strip():

        chunks.append(current.strip())

    return chunks

##Creating the Contract Chunk Table

The cleaned contract text is converted into a chunk-level DataFrame.

Each chunk receives a chunk ID and associated text.

This table becomes the searchable evidence space for the contract.

Instead of retrieving evidence from CUAD's pre-labeled clause snippets, MeridianIQ can now retrieve evidence from the actual raw contract being processed.

In [ ]:
chunks_df=chunk_text(clean_text)
chunks_df.shape

(109, 2)

In [ ]:
chunks_df.head()

,chunk_id,chunk_text
0,0,"ALAMOGORDO FINANCIAL CORPORATION 1,101,643 Shares COMMON STOCK (Par Value $.0l Per Share) Subscription Price $10.00 Per Share AGENCY AGREEMENT ___________ __, 2000 Charles Webb & Company, a Division of Keefe, Bruyette & Woods, Inc. 211 Bradenton Avenue Dublin, Ohio 43017 Ladies and Gentlemen: Al..."
1,1,"In accordance with the Stock Issuance Plan adopted by its Board of Directors (the ""Plan""), the Company will offer and sell up to 1,101,643 shares of its common stock, par value, $.01 per share (the ""Shares"" or ""Common Stock""), in a subscription offering (the ""Subscription Offering"") to (1) depos..."
2,2,"To the extent Shares remain unsold in the Subscription Offering, the Company is offering for sale in a community offering (the ""Community Offering"" and when referred to together with the Subscription Offering, the ""Subscription and Community Offering"") the Shares not so subscribed for or ordered..."
3,3,"It is acknowledged that the purchase of Shares in the Offering is subject to the maximum and minimum purchase limitations as described in the Plan and that the Company and the Bank may reject, in whole or in part, any orders received in the Community Offering or Syndicated Community Offering. Th..."
4,4,"The prospectus, as amended, on file with the Commission at the time the Registration Statement initially became effective is hereinafter called the ""Prospectus,"" except that if any Prospectus is filed by the Company pursuant to Rule 424(b) or (c) of the rules and regulations of the Commission un..."


##Preparing Full Text for Clause Detection

The trained TF-IDF detector expects contract-level text as input.

For clause prediction, MeridianIQ uses the cleaned full contract text.

This allows the model to scan the complete contract and predict which MVP clauses are likely present.

This step bridges raw text ingestion with the clause detection model trained earlier.

In [ ]:
x_contract=[clean_text]
x_tfidf=tfidf.transform(x_contract)
clause_predictions=clause_detector.predict(x_tfidf)
clause_predictions.shape

(1, 23)

##Creating the Predicted Contract Fingerprint

The raw model predictions are converted into a readable contract fingerprint.

Each MVP clause becomes a column indicating whether that clause was detected.

This predicted fingerprint is important because it becomes the input for the risk engine and report generation workflow.

In [ ]:
label_cols=[
    to_snake_case(clause)
    for clause in mvp_config["clause_name"].tolist()
]

predicted_fingerprint=pd.DataFrame(
    clause_predictions,
    columns=label_cols
)

predicted_fingerprint.insert(0,"filename",os.path.basename(sample_txt_path))
predicted_fingerprint

,filename,non_compete,exclusivity,no_solicit_of_customers,no_solicit_of_employees,termination_for_convenience,change_of_control,anti_assignment,revenue_profit_sharing,price_restrictions,minimum_commitment,volume_restriction,ip_ownership_assignment,joint_ip_ownership,license_grant,unlimited_all_you_can_eat_license,irrevocable_or_perpetual_license,post_termination_services,audit_rights,uncapped_liability,cap_on_liability,liquidated_damages,warranty_duration,insurance
0,ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt,1,1,0,0,0,1,1,1,0,1,0,1,0,1,0,0,1,1,0,1,1,0,1


##Converting Predictions into Readable Clause Results

Model outputs are useful for computation, but users need readable clause names.

This step converts the predicted fingerprint into a clean table showing:

* Clause name
* Clause column
* Detection status

This table makes the model output easier to inspect and supports later reporting.

In [ ]:
detected_clauses=[]

for clause_name,col in zip(mvp_config["clause_name"],label_cols):
  detected=bool(predicted_fingerprint.loc[0,col])

  detected_clauses.append({
      "clause_name":clause_name,
      "clause_column":col,
      "detected":detected
  })

detected_clauses_df=pd.DataFrame(detected_clauses)
detected_clauses_df[detected_clauses_df["detected"]==True]


,clause_name,clause_column,detected
0,Non-Compete,non_compete,True
1,Exclusivity,exclusivity,True
5,Change Of Control,change_of_control,True
6,Anti-Assignment,anti_assignment,True
7,Revenue/Profit Sharing,revenue_profit_sharing,True
9,Minimum Commitment,minimum_commitment,True
11,Ip Ownership Assignment,ip_ownership_assignment,True
13,License Grant,license_grant,True
16,Post-Termination Services,post_termination_services,True
17,Audit Rights,audit_rights,True


##Loading the Semantic Embedding Model

This step loads the Sentence Transformer model used for semantic evidence retrieval.

Although TF-IDF performed better for clause detection, Sentence Transformers are still valuable for retrieval because they compare meaning rather than exact wording.

This is where MeridianIQ's hybrid architecture becomes visible:

TF-IDF for clause detection
Sentence Transformers for evidence retrieval

In [ ]:
embedding_model_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

##Embedding Contract Chunks

Each contract chunk is converted into a dense semantic embedding.

In [ ]:
chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

##Building Queries for Detected Clauses

MeridianIQ only retrieves evidence for clauses that were detected as present.

For each detected clause, the system builds a semantic query using the clause name and its business description from the risk configuration.

Using both the clause name and description makes the query richer and improves the chance of retrieving relevant evidence.

In [ ]:
detected_clause_names = detected_clauses_df[
    detected_clauses_df["detected"] == True
]["clause_name"].tolist()

detected_config = mvp_config[
    mvp_config["clause_name"].isin(detected_clause_names)
].copy()

clause_queries = []

for _, row in detected_config.iterrows():
    query = f"{row['clause_name']}. {row['business_description']}"
    clause_queries.append({
        "clause_name": row["clause_name"],
        "query": query
    })

clause_queries_df = pd.DataFrame(clause_queries)

clause_queries_df.head()

,clause_name,query
0,Non-Compete,"Non-Compete. Restricts a party from competing in certain markets, geographies, or sectors."
1,Exclusivity,Exclusivity. Creates exclusive dealing obligations that may limit business flexibility.
2,Change Of Control,Change Of Control. Creates rights or obligations triggered by ownership or control changes.
3,Anti-Assignment,Anti-Assignment. Restricts transfer or assignment of contractual rights.
4,Revenue/Profit Sharing,Revenue/Profit Sharing. Requires sharing revenue or profit with the counterparty.


##Computing Query-to-Chunk Similarity

Detected clause queries are embedded and compared against all contract chunk embeddings using cosine similarity.

In [ ]:
query_embeddings = embedding_model.encode(
    clause_queries_df["query"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

similarity_matrix = cosine_similarity(query_embeddings, chunk_embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

##Retrieving Evidence Chunks

For each detected clause, MeridianIQ retrieves the top matching contract chunks.

Each evidence record contains:

* Contract filename
* Clause name
* Evidence rank
* Similarity score
* Chunk ID
* Evidence text

This creates a grounded explanation layer for the raw TXT pipeline.

In [ ]:
top_k = 3

evidence_records = []

for query_idx, row in clause_queries_df.iterrows():
    scores = similarity_matrix[query_idx]
    top_indices = np.argsort(scores)[::-1][:top_k]

    for rank, chunk_idx in enumerate(top_indices, start=1):
        chunk = chunks_df.iloc[chunk_idx]

        evidence_records.append({
            "filename": os.path.basename(sample_txt_path),
            "clause_name": row["clause_name"],
            "rank": rank,
            "similarity_score": scores[chunk_idx],
            "chunk_id": chunk["chunk_id"],
            "evidence_text": chunk["chunk_text"]
        })

evidence_df = pd.DataFrame(evidence_records)

evidence_df.head()

,filename,clause_name,rank,similarity_score,chunk_id,evidence_text
0,ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt,Non-Compete,1,0.333786,2,"To the extent Shares remain unsold in the Subscription Offering, the Company is offering for sale in a community offering (the ""Community Offering"" and when referred to together with the Subscription Offering, the ""Subscription and Community Offering"") the Shares not so subscribed for or ordered..."
1,ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt,Non-Compete,2,0.311021,17,"(e) The MHC is and, as of the Closing Date, will continue to be duly organized and validly existing as a federally chartered mutual holding company under the laws of the United States, duly authorized to conduct its business and own its property as described in the Registration Statement and the..."
2,ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt,Non-Compete,3,0.305842,19,"(i) The Bank has been organized and is a validly existing federally chartered savings and loan association in capital stock form of organization, duly authorized to conduct its business and own its property as described in the Registration Statement and the Prospectus; the Bank has obtained all ..."
3,ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt,Exclusivity,1,0.473883,77,"In addition, such opinion may be limited to present statutes, regulations and judicial interpretations and to facts as they presently exist; in rendering such opinion, such counsel need assume no obligation to revise or supplement it should the present laws be changed by legislative or regulator..."
4,ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt,Exclusivity,2,0.446971,25,"The consummation of the Offering, the execution, delivery and performance of this Agreement and the consummation of the transactions herein contemplated have been duly and validly authorized by all necessary corporate action on the part of the Company, the MHC and the Bank and this Agreement has..."


##Exporting Raw TXT Pipeline Outputs

In [ ]:
predicted_fingerprint.to_csv("sample_predicted_fingerprint.csv",index=False)
detected_clauses_df.to_csv("sample_detected_clauses.csv",index=False)
evidence_df.to_csv("sample_evidence_chunks.csv",index=False)